# EchoVerify: Training, Evaluation & Screening Walkthrough

**Robust Deep Learning for Audio Deepfake Detection: Silence-Shortcut Audit, Compression Robustness, and Practical Screening**

This notebook demonstrates the end-to-end EchoVerify pipeline using the modular `src/` codebase:
1. **Protocol Loading & Stratified Manifests:** Load official ASVspoof 2019 LA protocols and create deterministic cohorts (seed 42) preserving class balance and attack coverage (`A01`–`A06`).
2. **Acoustic Preprocessing:** 16 kHz mono conversion, 4-second cropping/padding (64,000 samples), and leading/trailing silence edge auditing.
3. **CNN Architecture & Feature Extraction:** Log-Mel spectrogram generation (64 Mel bands, $n_{fft}=512$, $hop=160$) and compact 4-block Conv2D network with class-weighted BCE loss.
4. **Benchmark Evaluation:** Calculate Accuracy, Precision, Recall, F1, ROC-AUC, EER, and Genuine Speech False Positive Rate (FPR) using development-locked operating thresholds.
5. **Screening Diagnostics & Visualizations:** Waveform inspection, log-Mel spectrograms, training curves, and confusion matrices.

> **Reproducibility Note:** When running on HiPerGator or a local workstation, set `ASVSPOOF2019_LA_ROOT` to the directory containing `ASVspoof2019_LA_cm_protocols/` and audio folders.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure repository root is on PYTHONPATH
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.data.protocol import ASVSpoofProtocol, resolve_dataset_root
from src.data.dataset import create_stratified_manifest, get_manifest_statistics, ASVSpoofDataset
from src.data.preprocessing import AudioPreprocessor, PreprocessConfig
from src.models.cnn import EchoVerifyCNN
from src.evaluation.metrics import compute_all_metrics, compute_eer
from src.evaluation.visualizer import plot_confusion_matrix, plot_training_curves

print(f"Repository root: {REPO_ROOT}")
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

## 1. Locate Dataset & Check Official Protocols

In [ ]:
try:
    data_root = resolve_dataset_root()
    protocol = ASVSpoofProtocol(data_root=data_root)
    print(f"Found ASVspoof 2019 LA at: {data_root}")
    print("Train split available:", protocol.is_split_available("train"))
    print("Dev split available:  ", protocol.is_split_available("dev"))
    print("Eval split available: ", protocol.is_split_available("eval"))
except FileNotFoundError as e:
    print(f"Dataset check: {e}")
    print("Set ASVSPOOF2019_LA_ROOT before running dataset-dependent cells.")

## 2. Generate Deterministic Stratified Manifests (Seed 42)
Subsamples up to 2,000 training clips and 1,000 development clips while preserving class balance and attack ID representation (`A01`–`A06`).

In [ ]:
# Load official protocols
train_df = protocol.load_split("train", validate_audio=True)
dev_df = protocol.load_split("dev", validate_audio=True)

# Sample stratified cohorts with seed 42
manifest_train = create_stratified_manifest(train_df, max_samples=2000, seed=42)
manifest_dev = create_stratified_manifest(dev_df, max_samples=1000, seed=42)

train_stats = get_manifest_statistics(manifest_train)
dev_stats = get_manifest_statistics(manifest_dev)

print("Sampled Training Cohort Summary:")
print(train_stats)
print("\nSampled Development Cohort Summary:")
print(dev_stats)

## 3. Shared Preprocessing & Silence Edge Inspection
Compares raw versus leading/trailing silence-trimmed audio standardization.

In [ ]:
raw_config = PreprocessConfig(mode="raw", sample_rate=16000, target_duration_sec=4.0)
trimmed_config = PreprocessConfig(mode="trimmed", trim_top_db=30.0, sample_rate=16000, target_duration_sec=4.0)

raw_preprocessor = AudioPreprocessor(raw_config)
trimmed_preprocessor = AudioPreprocessor(trimmed_config)

# Process a sample audio clip
sample_audio_path = manifest_dev.iloc[0]["audio_path"]
raw_wave, sil_meta = raw_preprocessor.process_waveform(sample_audio_path)
trimmed_wave, _ = trimmed_preprocessor.process_waveform(sample_audio_path)

print(f"Processed audio: {sample_audio_path.name}")
print(f"Raw waveform shape: {raw_wave.shape} (target samples: {raw_config.target_samples})")
print(f"Silence Edge Audit: Leading={sil_meta['lead_silence_sec']:.3f}s, Trailing={sil_meta['trail_silence_sec']:.3f}s, Active={sil_meta['active_sec']:.3f}s")

## 4. CNN Architecture & Log-Mel Spectrogram Verification

In [ ]:
model = EchoVerifyCNN(
    sample_rate=16000,
    n_mels=64,
    n_fft=512,
    hop_length=160,
)

# Pass dummy 4-second batch through the model
dummy_batch = torch.randn(2, 64000)
dummy_logits = model(dummy_batch)
dummy_probs = model.predict_proba(dummy_batch)

print("Model Parameter Count:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Output Logits Shape:", dummy_logits.shape)
print("Sample Uncalibrated Scores:", dummy_probs.squeeze().tolist())

## 5. Development Metrics & Threshold Locking

In [ ]:
metrics_file = REPO_ROOT / "results" / "dev_metrics.json"
if metrics_file.exists():
    import json
    with open(metrics_file, "r") as f:
        dev_results = json.load(f)
    print("Loaded Development Benchmark Results:")
    print(json.dumps(dev_results, indent=2))
else:
    print(f"No previous benchmark results found at {metrics_file}.")
    print("Run `python src/train.py` to generate real benchmark metrics.")